In [50]:
import pandas as pd
from sklearn.metrics import classification_report, average_precision_score
from xgboost import XGBClassifier

In [51]:
full_grid = pd.read_csv("../data/taylor_swift_model_df.csv")

In [52]:

tour_order = [
    "Fearless", "Speak Now World Tour", "The Red Tour",
    "The 1989 World Tour", "reputation Stadium Tour", "The Eras Tour"
]

feature_cols = [
    "played_last_tour", "cumulative_play_rate", "song_age_tours",
    "is_new_album_song", "tour_type"
]
target_col = "was_played_on_tour"

In [53]:
full_grid_encoded = pd.get_dummies(full_grid, columns=["tour_type"], drop_first=True)
full_grid_encoded["is_new_album_song"] = full_grid_encoded["is_new_album_song"].astype(int)

feature_cols_encoded = [c for c in full_grid_encoded.columns if c in feature_cols or c.startswith("tour_type_")]
print(feature_cols_encoded)

['played_last_tour', 'cumulative_play_rate', 'song_age_tours', 'is_new_album_song', 'tour_type_single_album']


In [54]:
results_xgb = {}
for held_out_tour in tour_order:
    train = full_grid_encoded[full_grid_encoded["tour"] != held_out_tour]
    test = full_grid_encoded[full_grid_encoded["tour"] == held_out_tour]
    
    X_train, y_train = train[feature_cols_simplified], train[target_col]
    X_test, y_test = test[feature_cols_simplified], test[target_col]
    
    model = XGBClassifier(eval_metric="logloss", random_state=42)
    model.fit(X_train, y_train)
    
    probs = model.predict_proba(X_test)[:, 1]
    pr_auc = average_precision_score(y_test, probs)
    results_xgb[held_out_tour] = pr_auc
    print(f"Held out: {held_out_tour:30s} PR-AUC: {pr_auc:.3f}")

print("\nAverage PR-AUC:", sum(results_xgb.values()) / len(results_xgb))

Held out: Fearless                       PR-AUC: 1.000
Held out: Speak Now World Tour           PR-AUC: 0.990
Held out: The Red Tour                   PR-AUC: 0.810
Held out: The 1989 World Tour            PR-AUC: 0.932
Held out: reputation Stadium Tour        PR-AUC: 0.748
Held out: The Eras Tour                  PR-AUC: 0.957

Average PR-AUC: 0.9061839694705509


In [55]:
model_full = XGBClassifier(eval_metric="logloss", random_state=42)
model_full.fit(full_grid_encoded[feature_cols_simplified], full_grid_encoded[target_col])

importances = pd.Series(model_full.feature_importances_, index=feature_cols_simplified).sort_values(ascending=False)
print(importances)

song_age_tours            0.790173
tour_type_single_album    0.120531
played_last_tour          0.089296
is_new_album_song         0.000000
dtype: float32


In [56]:
full_grid_encoded.groupby("is_new_album_song")["song_age_tours"].describe()

,count,mean,std,min,25%,50%,75%,max
is_new_album_song,,,,,,,,
0,1189.0,1.455845,1.429880,0.0,0.0,1.0,2.0,5.0
1,66.0,0.030303,0.246183,0.0,0.0,0.0,0.0,2.0


In [57]:
full_grid_encoded.groupby("song_age_tours")["was_played_on_tour"].mean()

song_age_tours
0.0    1.000000
1.0    0.204724
2.0    0.218182
3.0    0.227273
4.0    0.274336
5.0    0.720000
Name: was_played_on_tour, dtype: float64

In [58]:
full_grid_encoded[(full_grid_encoded["song_age_tours"] == 5)].groupby("tour")["was_played_on_tour"].mean()

tour
The Eras Tour    0.72
Name: was_played_on_tour, dtype: float64

NLP

In [75]:
import lyricsgenius
from dotenv import load_dotenv
import os
import json
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import re
import requests
import base64
import time

In [ ]:
# load_dotenv(dotenv_path="../.env")
# genius = lyricsgenius.Genius(os.getenv("GENIUS_ACCESS_TOKEN"), timeout=15, retries=3)
# genius.verbose = False

# all_songs = full_grid_encoded["song_name"].unique()

# lyrics_data = {}
# for song in all_songs:
#     try:
#         song_result = genius.search_song(song, "Taylor Swift")
#         if song_result:
#             lyrics_data[song] = song_result.lyrics
#         else:
#             lyrics_data[song] = None
#     except Exception as e:
#         print(f"Failed on {song}: {e}")
#         lyrics_data[song] = None

# print(f"Pulled lyrics for {sum(v is not None for v in lyrics_data.values())}/{len(all_songs)} songs")

Pulled lyrics for 482/489 songs


In [ ]:
#save pulled lyrics, so re-running will not be required
# with open("../data/taylor_swift_lyrics.json", "w") as f:
#     json.dump(lyrics_data, f)

In [ ]:
'''
uses VADER, a sentiment scoring tool
it is used to score the mood of the song
output: score from -1 to 1
    -1 represents negative (melancholy) lyrics
    1 represents positive (upbeat)lyrics
'''
analyzer = SentimentIntensityAnalyzer()

sentiment_scores = {}
for song, lyrics in lyrics_data.items():
    if lyrics:
        scores = analyzer.polarity_scores(lyrics)
        sentiment_scores[song] = scores["compound"]  # -1 (very negative) to +1 (very positive)
    else:
        sentiment_scores[song] = None

In [ ]:
def compute_lyric_features(lyrics):
    '''
    this function computes 3 features
    1. average sentence length 
        purpose: to characterise a song as one made of short, punchy phrases
        or longer flowing sentences like a narrative
        output: a single number (float) per song
    2. proper noun density
        purpose: how specific a song's lyrics are, 
            especially important since Swift uses many references such as "Dear John"
        output: a single number (small float) per song
    3. unique word ratio
        purpose: how repetitive a song's lyrics are
        output: a decimal between 0 and 1 (ratio closer to 1 means more uniqueness)
    '''
    if not lyrics:
        return None, None, None
    words = lyrics.split()
    sentences = re.split(r'[.!?]', lyrics)
    avg_sentence_len = len(words) / max(len(sentences), 1)
    
    # proper noun density proxy: capitalized words not at sentence-start (rough heuristic)
    proper_noun_count = len(re.findall(r'(?<!\. )(?<!^)[A-Z][a-z]+', lyrics))
    proper_noun_density = proper_noun_count / max(len(words), 1)
    
    unique_word_ratio = len(set(w.lower() for w in words)) / max(len(words), 1)
    
    return avg_sentence_len, proper_noun_density, unique_word_ratio

complexity_data = {song: compute_lyric_features(lyrics) for song, lyrics in lyrics_data.items()}

In [64]:
lyric_features_df = pd.DataFrame([
    {
        "song_name": song,
        "sentiment": sentiment_scores.get(song),
        "avg_sentence_len": complexity_data[song][0] if complexity_data[song] else None,
        "proper_noun_density": complexity_data[song][1] if complexity_data[song] else None,
        "unique_word_ratio": complexity_data[song][2] if complexity_data[song] else None,
    }
    for song in all_songs
])

full_grid_encoded = full_grid_encoded.merge(lyric_features_df, on="song_name", how="left")

In [ ]:
#fill in missing values
full_grid_encoded["sentiment"] = full_grid_encoded["sentiment"].fillna(0)  # neutral sentiment default
full_grid_encoded["avg_sentence_len"] = full_grid_encoded["avg_sentence_len"].fillna(full_grid_encoded["avg_sentence_len"].mean())
full_grid_encoded["proper_noun_density"] = full_grid_encoded["proper_noun_density"].fillna(full_grid_encoded["proper_noun_density"].mean())
full_grid_encoded["unique_word_ratio"] = full_grid_encoded["unique_word_ratio"].fillna(full_grid_encoded["unique_word_ratio"].mean())

print(full_grid_encoded.isna().sum().sum())  # should be 0 across the whole df now

0


In [69]:
feature_cols_nlp = [
    "played_last_tour", "song_age_tours", "tour_type_single_album",
    "sentiment", "avg_sentence_len", "proper_noun_density", "unique_word_ratio"
]
results_nlp = {}
for held_out_tour in tour_order:
    train = full_grid_encoded[full_grid_encoded["tour"] != held_out_tour]
    test = full_grid_encoded[full_grid_encoded["tour"] == held_out_tour]
    
    X_train, y_train = train[feature_cols_nlp], train[target_col]
    X_test, y_test = test[feature_cols_nlp], test[target_col]
    
    model = XGBClassifier(eval_metric="logloss", random_state=42)
    model.fit(X_train, y_train)
    
    probs = model.predict_proba(X_test)[:, 1]
    pr_auc = average_precision_score(y_test, probs)
    results_nlp[held_out_tour] = pr_auc
    print(f"Held out: {held_out_tour:30s} PR-AUC: {pr_auc:.3f}")

print("\nAverage PR-AUC with NLP features:", sum(results_nlp.values()) / len(results_nlp))

Held out: Fearless                       PR-AUC: 1.000
Held out: Speak Now World Tour           PR-AUC: 0.998
Held out: The Red Tour                   PR-AUC: 0.937
Held out: The 1989 World Tour            PR-AUC: 0.943
Held out: reputation Stadium Tour        PR-AUC: 0.850
Held out: The Eras Tour                  PR-AUC: 0.971

Average PR-AUC with NLP features: 0.950120631371474


In [70]:
model_full_nlp = XGBClassifier(eval_metric="logloss", random_state=42)
model_full_nlp.fit(full_grid_encoded[feature_cols_nlp], full_grid_encoded[target_col])

importances_nlp = pd.Series(model_full_nlp.feature_importances_, index=feature_cols_nlp).sort_values(ascending=False)
print(importances_nlp)

song_age_tours            0.561393
played_last_tour          0.102788
tour_type_single_album    0.100404
unique_word_ratio         0.072832
avg_sentence_len          0.063834
proper_noun_density       0.058412
sentiment                 0.040337
dtype: float32


In [ ]:
full_grid_encoded.to_csv("../data/taylor_swift_model_df_final_nlp.csv", index=False)

['../data/xgb_model_final_nlp.pkl']